# LCO‑HBI Analysis (Fixed Copy)
This simplified, cleaned notebook loads a solved PyPSA network, inspects `n.statistics()` output, and computes a defensible LCO‑HBI using a robust fallback if statistics are unavailable.

In [10]:
import os
import pathlib
from datetime import datetime
import pypsa

print("Run at", datetime.now().isoformat() + "Z")
print("pypsa version", getattr(pypsa, "__version__", "unknown"))

Run at 2026-05-20T15:13:42.482589Z
pypsa version 1.1.2


In [11]:
repo_root = pathlib.Path("../..").resolve()
relative_network_path = "resources/lco-hbi/cost_year~2050/Europe_reserved/network_10.nc"

NETWORK_PATH = repo_root / relative_network_path
DRI_LINK_PATTERNS = ["dri", "hbi"]
DEFAULT_WACC = 0.07
DEFAULT_LIFETIME_YEARS = 20
FIGURES_DIR = "figures"
OUTPUTS_DIR = "outputs"
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
print("Config loaded. NETWORK_PATH=", NETWORK_PATH)

Config loaded. NETWORK_PATH= C:\Users\JanLeopoldTautorus\Repos\shift\resources\lco-hbi\cost_year~2050\Europe_reserved\network_10.nc


In [12]:
# Load network
p = pathlib.Path(NETWORK_PATH)
if not p.exists():
    raise FileNotFoundError(f"Network file not found: {p}")
n = pypsa.Network(str(p))
print("Network loaded; snapshots:", len(getattr(n, "snapshots", [])))

INFO:pypsa.network.io:New version 1.2.1 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'LCOX-Europe-hbi-10.0' has buses, carriers, generators, links, loads, stores, sub_networks


Network loaded; snapshots: 8760


In [23]:
n.stats.capex()

component  carrier                 
Generator  renewable_pv                2.156046e+08
           renewable_wind_offshore     8.000000e-05
           renewable_wind_onshore      2.275435e+09
Link       direct_reduction_furnace    9.958061e+08
           electrolysis                4.147767e+08
Store      hydrogen                    4.193447e+07
dtype: float64

In [17]:
n.stats.expanded_capacity()

component  carrier                 
Generator  -                                0.00000
           battery                          0.00000
           grid_electricity                 0.00000
           renewable_pv                  5731.44069
           renewable_wind_onshore       16673.82956
Link       direct_reduction_furnace      1810.48216
           electrolysis                  7445.48173
Store      hbi                         192304.80782
           hydrogen                    261336.41259
dtype: float64

In [18]:
n.stats.optimal_capacity()

component  carrier                 
Generator  -                           1.000000e+10
           battery                     1.000000e+10
           grid_electricity            1.000000e+10
           renewable_pv                5.731441e+03
           renewable_wind_onshore      1.667383e+04
Link       direct_reduction_furnace    1.810482e+03
           electrolysis                7.445482e+03
Store      hbi                         1.923048e+05
           hydrogen                    2.613364e+05
dtype: float64

In [19]:
n.stats.overnight_cost()

component  carrier                 
Generator  renewable_pv                2.108408e+09
           renewable_wind_offshore     7.600000e-04
           renewable_wind_onshore      2.145033e+10
Link       direct_reduction_furnace    9.738038e+09
           electrolysis                4.056123e+09
Store      battery_elec                3.000000e-05
           hydrogen                    4.193143e+08
dtype: float64

In [28]:
n.stats.system_cost.iplot.bar()